# Lab: Multiple Linear Regression

---

## 1. Introduction
In the [previous lab](./03_lab--simple_linear_regression.ipynb), we built a Simple Linear Regression model using only one feature. Now we'll take the next step by building a **Multiple Linear Regression** model, which can leverage two or more features to create a more powerful and nuanced prediction.

* **Goal:** To build a model that predicts the CO2 emissions of a car using multiple features, such as engine size and fuel consumption.
* **Data:** We will continue using the Canadian fuel consumption dataset.

---

## 2. Setup and Data Loading

As before, we'll start by importing our essential libraries and loading the dataset.

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the data
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%202/data/FuelConsumptionCo2.csv"
df = pd.read_csv(url)

# Display a sample of the data
df.sample(5)

,MODELYEAR,MAKE,MODEL,VEHICLECLASS,ENGINESIZE,CYLINDERS,TRANSMISSION,FUELTYPE,FUELCONSUMPTION_CITY,FUELCONSUMPTION_HWY,FUELCONSUMPTION_COMB,FUELCONSUMPTION_COMB_MPG,CO2EMISSIONS
558,2014,JAGUAR,XJL AWD PORTFOLIO,FULL-SIZE,3.0,6,AS8,Z,14.7,9.6,12.4,23,285
1029,2014,VOLKSWAGEN,EOS,SUBCOMPACT,2.0,4,A6,Z,10.9,8.0,9.6,29,221
990,2014,TOYOTA,RAV4,SUV - SMALL,2.5,4,AS6,X,10.0,7.6,8.9,32,205
107,2014,BMW,650i xDRIVE COUPE,COMPACT,4.4,8,A8,Z,14.4,9.6,12.2,23,281
469,2014,GMC,YUKON,SUV - STANDARD,5.3,8,A6,X,16.0,11.1,13.8,20,317



---

## 3. Feature Selection for Multiple Regression

With multiple regression, choosing the right set of features is crucial. We want features that are highly correlated with our target (`CO2EMISSIONS`) but **not highly correlated with each other**. Including features that are correlated with each other (a problem called **multicollinearity**) can make the model unstable and difficult to interpret.

Let's start by creating a subset of potentially useful numerical features.

In [2]:
cdf = df[['ENGINESIZE','CYLINDERS','FUELCONSUMPTION_CITY','FUELCONSUMPTION_HWY','FUELCONSUMPTION_COMB','FUELCONSUMPTION_COMB_MPG','CO2EMISSIONS']]
cdf.head()

,ENGINESIZE,CYLINDERS,FUELCONSUMPTION_CITY,FUELCONSUMPTION_HWY,FUELCONSUMPTION_COMB,FUELCONSUMPTION_COMB_MPG,CO2EMISSIONS
0,2.0,4,9.9,6.7,8.5,33,196
1,2.4,4,11.2,7.7,9.6,29,221
2,1.5,4,6.0,5.8,5.9,48,136
3,3.5,6,12.7,9.1,11.1,25,255
4,3.5,6,12.1,8.7,10.6,27,244


Now, let's generate a **correlation matrix**. This is a powerful tool that shows the pairwise correlation between all of our selected features. Values close to 1.0 or -1.0 indicate a strong correlation.

In [4]:
cdf.corr().round(2)

,ENGINESIZE,CYLINDERS,FUELCONSUMPTION_CITY,FUELCONSUMPTION_HWY,FUELCONSUMPTION_COMB,FUELCONSUMPTION_COMB_MPG,CO2EMISSIONS
ENGINESIZE,1.00,0.93,0.83,0.78,0.82,-0.81,0.87
CYLINDERS,0.93,1.00,0.80,0.72,0.78,-0.77,0.85
FUELCONSUMPTION_CITY,0.83,0.80,1.00,0.97,1.00,-0.94,0.90
FUELCONSUMPTION_HWY,0.78,0.72,0.97,1.00,0.99,-0.89,0.86
FUELCONSUMPTION_COMB,0.82,0.78,1.00,0.99,1.00,-0.93,0.89
FUELCONSUMPTION_COMB_MPG,-0.81,-0.77,-0.94,-0.89,-0.93,1.00,-0.91
CO2EMISSIONS,0.87,0.85,0.90,0.86,0.89,-0.91,1.00


### How to Read the Correlation Matrix

1. **Focus on the Target Row (`CO2EMISSIONS`):** Look at